# params-iterable-vs-groups — worked example 2: Generator params materialized safely before dispatch

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `params-iterable-vs-groups`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A generator can only be iterated once. If you peek at its first element by calling `next()` and then try to iterate again, the first element is lost. The safe pattern is to call `list(params)` upfront, converting the generator to a concrete list before any dispatch logic runs. This way, a single-use generator of model parameters works correctly even though the optimizer needs to inspect and store all the tensors.

## Worked solution

**Step 1 — Convert to list immediately.** The very first line is `materialized = list(params)`. Whether `params` is a list, tuple, or generator, we now have a plain list we can safely index and iterate multiple times.

**Step 2 — Empty guard.** If `materialized` is empty, PyTorch raises `ValueError('optimizer got an empty parameter list')`. We match this exactly.

**Step 3 — Dispatch on first element type.** `isinstance(materialized[0], t.Tensor)` catches all tensor subclasses (`nn.Parameter` included). If True, wrap as a single group.

**Step 4 — Verify that the generator was fully consumed.** We simulate the scenario by passing a generator expression and confirming the output contains all three tensors, proving the materialization captured everything before dispatch.

In [ ]:
import torch as t

def normalize_params_from_generator(params, default_lr):
    """Safely normalize params even if it is a single-use generator."""
    materialized = list(params)  # must come first — exhausts the generator once
    if not materialized:
        raise ValueError('optimizer got an empty parameter list')
    first = materialized[0]
    if isinstance(first, t.Tensor):
        return [{'params': materialized, 'lr': default_lr}]
    if isinstance(first, dict):
        out = []
        for g in materialized:
            gc = dict(g)
            if 'lr' not in gc:
                gc['lr'] = default_lr
            out.append(gc)
        return out
    raise TypeError(f'Unexpected element type: {type(first).__name__}')

# Build tensors
t.manual_seed(7)
w1 = t.randn(3, 3)
w2 = t.randn(3)
w3 = t.randn(5, 3)

# Pass as a generator — can only be iterated once
def param_gen():
    yield w1
    yield w2
    yield w3

groups = normalize_params_from_generator(param_gen(), default_lr=3e-4)
print(f'Groups: {len(groups)}')             # 1
print(f'Tensors captured: {len(groups[0]["params"])}')  # 3
print(f'All present: {groups[0]["params"][2] is w3}')   # True